In [16]:
import re
import joblib
import pandas as pd
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import resample

# Paths and constants
DATA_PATH = "complaints_dataset.csv"
MODEL_PATH = "priority_model.joblib"
RANDOM_STATE = 42
TEST_SIZE = 0.20
VALID_LOCATIONS = ["industrial", "commercial", "residential", "public park", "highway"]

# Text cleaning
def clean_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Load dataset
def load_dataset(path):
    df = pd.read_csv(path)
    df["text"] = df["description"].astype(str).apply(clean_text)
    df["location"] = df["location"].astype(str)
    df["duration_days"] = pd.to_numeric(df["days_open"], errors="coerce").fillna(0).astype(int)
    df["priority"] = df["priority"].astype(str)
    return df

# Balance dataset by upsampling minority classes
def balance_dataset(df):
    counts = df['priority'].value_counts()
    max_count = counts.max()
    df_list = []
    for label in counts.index:
        df_label = df[df.priority == label]
        if len(df_label) < max_count:
            df_label = resample(df_label, replace=True, n_samples=max_count, random_state=RANDOM_STATE)
        df_list.append(df_label)
    return pd.concat(df_list).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# Build ML pipeline
def build_pipeline():
    preprocessor = ColumnTransformer(
        [
            ("text", TfidfVectorizer(ngram_range=(1, 2), max_features=3000), "text"),
            ("location", OneHotEncoder(handle_unknown="ignore"), ["location"]),
            ("duration", StandardScaler(), ["duration_days"]),
        ]
    )
    model = Pipeline([
        ("prep", preprocessor),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=3000)),
    ])
    return model

# Train and evaluate model
def train_and_evaluate(df):
    df = balance_dataset(df)  # balance classes
    le = LabelEncoder()
    y = le.fit_transform(df["priority"])
    X = df[["text", "location", "duration_days"]]

    stratify_y = y if all(pd.Series(y).value_counts() > 1) else None

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=stratify_y
    )

    model = build_pipeline()
    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print("Accuracy:", acc)

    test_labels = [le.classes_[i] for i in sorted(set(y_test))]
    print(classification_report(y_test, pred, target_names=test_labels))

    joblib.dump({"pipeline": model, "label_encoder": le}, MODEL_PATH)
    print("Model saved →", MODEL_PATH)

    return model, le

# Rule-based emergency priority
def rule_priority(text, duration):
    text = clean_text(text)
    high_kw = ["fire", "electrical", "gas leak", "explosion", "accident",
               "sewage overflow", "building collapse", "tree blocking road",
               "wire down", "chemical spill"]
    for k in high_kw:
        if k in text:
            return "High"
    return None  # Let ML predict for all other cases

# Load saved model
def load_model():
    data = joblib.load(MODEL_PATH)
    return data["pipeline"], data["label_encoder"]

# Predict priority
def predict_priority(text, location, duration, model_tuple=None):
    forced = rule_priority(text, duration)
    if forced is not None:
        return forced

    if model_tuple is None:
        pipeline, le = load_model()
    else:
        pipeline, le = model_tuple

    df = pd.DataFrame([{"text": clean_text(text), "location": location, "duration_days": duration}])
    p = pipeline.predict(df)[0]
    return le.inverse_transform([p])[0]

# Main execution
if __name__ == "__main__":
    if not Path(DATA_PATH).exists():
        print("Dataset missing:", DATA_PATH)
        exit()

    df = load_dataset(DATA_PATH)
    model, le = train_and_evaluate(df)

    print("\n--- Prediction Demo ---")
    while True:
        c = input("Complaint: ").strip()
        if not c:
            print("Complaint cannot be empty!")
            continue

        l = input(f"Location ({', '.join(VALID_LOCATIONS)}): ").strip().lower()
        if l not in VALID_LOCATIONS:
            print(f"Invalid location '{l}'. Please enter a valid location.")
            continue

        try:
            d = int(input("Duration days: ").strip())
            if d < 0:
                print("Duration cannot be negative!")
                continue
        except:
            print("Invalid duration! Enter a number.")
            continue

        priority = predict_priority(c, l, d, (model, le))
        print("Predicted Priority:", priority)
        break


Accuracy: 0.9693251533742331
              precision    recall  f1-score   support

        High       0.97      0.95      0.96        41
         Low       0.97      0.97      0.97        40
      Medium       0.97      0.95      0.96        41
         nan       0.95      1.00      0.98        41

    accuracy                           0.97       163
   macro avg       0.97      0.97      0.97       163
weighted avg       0.97      0.97      0.97       163

Model saved → priority_model.joblib

--- Prediction Demo ---
Complaint: Blocked drain
Location (industrial, commercial, residential, public park, highway): commercial
Duration days: 59
Predicted Priority: Medium
